# **Basic Data Visualisation**, **How to make tables in Python** - Code snippets

The following code relates to the **How to make tables in Python** section of the **Basic Data Visualisation** learning resource on the [Analysis for Action]([PLACEHOLDER]) platform.  

Use this notebook to explore and run the code snippets provided in the learning resource.

## Troubleshooting

If you encounter any issues running the code snippets, consider the following troubleshooting steps:
- Ensure you have followed the setup instructions in the [README.md](../../../README.md) file including `pip install requirements.txt` for required packages.
- Run code from top to bottom to ensure all dependencies are met.
- Clear all outputs and restart your Python kernel.
- For more detailed troubleshooting guidance, refer to the [troubleshooting document]([PLACEHOLDER]).

## Setup

Before running the code snippets, ensure you have;
- followed the setup instructions in the [README.md](../../../README.md) file,
- **[add further instructions as needed]**,
- run the code block below.

In [ ]:

# Imports
import os
import pandas as pd
import polars as pl
from great_tables import GT, vals, loc, style


# Set working directory to the learning_resources folder
repo_root = os.path.abspath(
    os.path.join(os.path.dirname(os.path.abspath("making_tables_in_python.ipynb")), ("../../../../.."))
)

print("repo_root:", repo_root)

learning_resources_dir = os.path.join(repo_root, "learning_resources")
os.chdir(learning_resources_dir)
print("Current working directory:", os.getcwd())

# Load data
vulnerable_path = os.path.join(learning_resources_dir, "data", "vulnerable.csv")
vuln = pd.read_csv(vulnerable_path)




## Introduction to Great Tables

The main package you will use to create a table in this file is great_tables. Similarly to matplotlib and seaborn, which are used to create charts, it uses a layered style and renders outputs to HTML (or images if preferred).

great_tables uses the following structure:

* a table header - which contains a title and subtitle
* a stub and stub head - which contains row labels
* column labels - which contain column labels
* the table body - which contains columns and rows of cells
* and the table footer - which contains footnotes and source notes

Note that to align with the best practice presented in the main guidance, some great_tables components will not be used. For example, it is recommended to include titles for data visualisations in body text rather than in images.

## Prepare your data

This code uses the 'vulnerable' dataset. 

The raw data is too large to use in a demonstration table. You need to filter it to make a smaller dataframe first.



In [ ]:

# Prepare the data to make it suitable for a table

vuln_for_select_countries_3y = vuln.loc[
    vuln["year"].isin([1997, 2002,2007]),
    ["country", "continent", "year", "vulnerable_pop"]
]

# Reshaping the data

vuln_for_select_countries_3y = vuln_for_select_countries_3y.pivot_table(
    values="vulnerable_pop",
    index=["country", "continent"],
    columns="year"
)

  # Selecting the top 2 values

vuln_for_select_countries_3y = (
    vuln_for_select_countries_3y

    
    .sort_values(by=["continent", 2007], ascending=False)
    .groupby("continent", group_keys=False)
    .head(2)
).reset_index()

# Change float values to integers
cols = [1997, 2002, 2007]
vuln_for_select_countries_3y[cols] = (
    vuln_for_select_countries_3y[cols].round(0).astype(int)
)

# To display the data

vuln_for_select_countries_3y



## Make the base table

Use the GT class to create a table.

The best practice discussed in the main guidance suggests grouping rows where appropriate.  In GT, rows are not grouped automatically, so grouping (for example by continent) must be specified explicitly.


In [ ]:
# Make a basic table
# Creating a GT table
vuln_table = GT(vuln_for_select_countries_3y)

vuln_table


In [ ]:

# If you want to show data grouped by continent
vuln_table_grouped_continent =  GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")

vuln_table_grouped_continent

Once you have created the GT table object, you can apply styling and formatting methods to customise it.

The sections below look at how to make your table align with the best practice discussed in the main guidance.


## Add labels

You can use a ‘spanner’ and ‘stubhead’ to label your columns in a GT table.

Should the columns not be adjacent to each other, tab_spanner() will automatically gather them together.


In [ ]:
# Label your columns

vuln_table_labels = (
        GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")

    # Add a stub head heading
    .tab_stubhead(label="Continent")

    # Add a spanner heading
    .tab_spanner(
    label="Years",
    columns= ["1997", "2002", "2007"]
))

vuln_table_labels

## Right-align figures

Use cols_align() to set the alignment of columns. 

Best practice is to align figures to the right, and fortunately, gt already does this by default. Below is some code if you ever need to change this.

In [ ]:
# Right align a column’s text


vuln_table = ( GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")
    
    .cols_align(
        align = "right", 
        columns = ["1997", "2002", "2007"]
)
)


vuln_table

## Add commas, decimals, and suffixes

You can use the fmt_*() functions to format the data in your table, including adding commas and specifying how many decimals you want to include.

Some examples are shown below. Refer to the fmt documentation for a full list:

•	fmt_number()
•	fmt_integer()
•	fmt_date()
•   fmt_time()
•	fmt_currency()
•	fmt_scientific()

You can also us the general fmt() which provides greater control in formatting raw data values than any of the specialized fmt_*().

Note that decimals do not apply to the vulnerable dataframe as all the figures are already integers (i.e. full numbers). However, you can try adapting this code and applying it to a dataframe that does include more than two decimals to see it in action.



In [ ]:
# Format numbers to include commas

vuln_table_comma =(
  GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")

  .fmt_number(
    columns = ["1997", "2002", "2007"],
    decimals = 0,
    use_seps = True)
)

vuln_table_comma


When dealing with large numbers e.g. millions, you may wish to use a suffix to simplify your table. It is a good idea to include decimals if so. This can be done using the boolean argument for 'compact'.

In [ ]:
# Format numbers to include a suffix

vuln_table_suffix = (
   GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")
  
  .fmt_number(
      columns = ["1997", "2002", "2007"],
      decimals = 2,
      compact = True)
)

vuln_table_suffix

## Rearrange columns and rows

Use cols_move to rearrange columns.

On those occasions where you need to move columns this way or that way, you can make use of the cols_move() function. The movement procedure here takes one or more specified columns (in the columns argument) and places them to the right of a different column (the after argument). The ordering of the columns to be moved is preserved, as is the ordering of all other columns in the table.

You can also use cols_move_to_start() and cols_move to end() to easily move a set of columns to the beginning or end of the column series.


In [ ]:
# Method 1: Rearranging columns in a table
# Using cols_move()

vuln_table_rearrange_cols = (
   GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")
   
   .cols_move(columns = ["2002", "1997"], after = "2007")
)

vuln_table_rearrange_cols 


In [ ]:
# Method 2: Rearranging columns in a table
# Using cols_move_to_end()

vuln_table_rearrange_cols = (
   GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")
   
   .cols_move_to_end(columns = ["2002", "1997"])
)

vuln_table_rearrange_cols 

Use row_group_order() to rearrange rows. In the example below, the continent variable is used, but country could be used just as well.


In [ ]:
# Manually rearrange the order of rows

vuln_table_rearrange_rows = (
   GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")

   .row_group_order(groups = ["Africa",
                               "Asia",
                               "Americas",
                               "Europe",
                               "Oceania"])
)

vuln_table_rearrange_rows

You can also rearrange rows so they appear in ranked order, from lowest to highest values or vice versa, within a specific column. You can do this by changing the arguments used in `.sort_values()`.


## Add summary rows and columns

Use summary_rows() and grand_summary_rows() to add summary rows to your table.

If your data table is a pandas DataFrame, you can use lambda functions to create summary rows, as shown in the code below. 


In [ ]:

# Add summary rows

vuln_table_summary = ( GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent")
                      
.summary_rows(
        fns={"Total": lambda x: x[["1997", "2002", "2007"]].sum(),
             "Average": lambda x: x[["1997", "2002", "2007"]].mean()}
        )
)


vuln_table_summary

Alternatively, Great Tables can use Polars expressions in summary_rows() and grand_summary_rows(), but the underlying table must be a Polars DataFrame. 

You can check to see what type your DataFrame is by running type(). 

The example below is a pandas DataFrame. Therefore, the dataframe must be converted to a Polar DataFrame first in order to use the pl.col().

In [ ]:
type(vuln_for_select_countries_3y)

In [ ]:

vuln_for_select_countries_3y_pl = pl.from_pandas(vuln_for_select_countries_3y)

In [ ]:
vuln_for_select_countries_3y_pl = pl.from_pandas(vuln_for_select_countries_3y)

vuln_table_summary_pl = (
    GT(
        vuln_for_select_countries_3y_pl,
        rowname_col="country",
        groupname_col="continent"
    )
    .summary_rows(
        fns={
            "Total": pl.col(["1997", "2002", "2007"]).sum(),
            "Average": pl.col(["1997", "2002", "2007"]).mean()
        }
    )
)

vuln_table_summary_pl


Add grand summary rows to the GT table by applying the same methods as above:
- use lambda when working with a Pandas DataFrame
- use pl.col() when working with a Polars Dataframe

The summary rows incorporate all of the available data, regardless of whether some of the data are part of row groups.

In [ ]:
# Add grand summary rows 

vuln_table_summary_grand = (GT(vuln_for_select_countries_3y, rowname_col="country", groupname_col="continent") 
                      
.grand_summary_rows(
        fns={"Min": lambda df: df.min(numeric_only=True),
             "Max": lambda df: df.max(numeric_only=True),
             "Mean": lambda df: df.mean(numeric_only=True)}
    )
)

vuln_table_summary_grand

In [ ]:
# Add grand summary rows using pl.col()

vuln_for_select_countries_3y_pl = pl.from_pandas(vuln_for_select_countries_3y)


vuln_table_summary_grand_pl = (
    GT(vuln_for_select_countries_3y_pl,
        rowname_col="country",
        groupname_col="continent"
    )
    
    .grand_summary_rows(
        fns={"Min": pl.col("1997", "2002", "2007").min(),
             "Max": pl.col("1997", "2002", "2007").max(),
             "Mean": pl.col("1997", "2002", "2007").mean()}
    )
)

vuln_table_summary_grand_pl

To add a summary column, you need to modify the original dataframe, then build a table from the new dataframe. 

In [ ]:

# Add a column for mean average
# Make sure columns are numeric first (otherwise it won't be possible to calculate an average)

vuln_for_select_countries_3y_avg = vuln_for_select_countries_3y.copy()

vuln_for_select_countries_3y_avg["Average"] = ( 
    vuln_for_select_countries_3y_avg[[1997, 2002, 2007]]
    .mean(axis=1)
    .round(1)
)

vuln_table_summary_col = GT(vuln_for_select_countries_3y_avg, rowname_col="country", groupname_col="continent")

vuln_table_summary_col

## Change cell width

You can change the width of the cells in your table using the cols_width function.

In [ ]:
# Change the width of columns

vuln_table_wide = (
    GT(vuln_for_select_countries_3y,
        rowname_col="country",
        groupname_col="continent"
    )
    .cols_width(
        cases = {
            "1997": "100px",
            "2002": "100px",
            "2007": "100px",
            "country": "70px",
        }
        )
)

vuln_table_wide

## Change cell colours

You can change the colours in your table to help rows, columns, or cells stand out. However, make sure you still adhere to the contrast ratio requirements discussed in the guidance on accessibility.

The code below changes the colour of the column headers.

In [ ]:

# Change the colour of the header row and make the text bold white
vuln_table_colour = (
    GT(vuln_for_select_countries_3y, 
       rowname_col="country", 
       groupname_col="continent")
.tab_options(column_labels_background_color="#324027",column_labels_font_weight="bold")
)

vuln_table_colour


The code below highlights a specific row, in this case, the cell with the highest value.

In [ ]:
# Change the colour of the header row and highlight the cell with the highest value

vuln_table_highlight = (
    GT(vuln_for_select_countries_3y, 
       rowname_col="country", 
       groupname_col="continent")

.tab_options(column_labels_background_color="#324027",column_labels_font_weight="bold")

.tab_style(
    style=[
        style.fill(color="#D17E38"),
        style.text(style="bold")
    ],
    locations=loc.body(
        columns="1997",
        rows=lambda df: df["1997"] == df["1997"].max()
        )
        )
)
vuln_table_highlight.show(target = "browser")

## Export

Once you are happy with your table, you will need to export it. GT has a different export methods depending on where you need to display your table. gtsave() can produce PNG, PDF, JPEG, WebP files and write_raw_html() produces a HTML file. Great tables does not enable direct export of SVG files. 

Remember that for publications, exporting your table as an HTML file is best.

By default your outputs will be saved to a folder called 'TEMP' in your D: drive, but feel free to change this.

In [ ]:

# Export as HTML

vuln_table.write_raw_html(filename = "D:/TEMP/vuln_table.html") 

# Export as a PNG
# For reference, this is not the best format for tables, but it is possible to export a table as a PNG.

vuln_table.gtsave(filename = "D:/TEMP/vuln_table.png") 

# Export as a PDF

vuln_table.gtsave(filename = "D:/TEMP/vuln_table.pdf") 



## Exercise

Use this exercise to test what you’ve learned about making tables. You can adapt much of the code covered in this section, but some basic data manipulation will also be required which has not been covered here.

Create a table in Python. Use the vulnerable dataset as your input data.

For an extra challenge, consider setting yourself an hour’s time limit to help simulate an emergency.

The table should:

* Show the size of vulnerable populations in Argentina, Malawi, Nepal, and the United Kingdom for the years 1982, 1987, and 1992
* Include a summary column showing the mean average size of vulnerable populations across these three years
* Include a spanner heading above the 1982, 1987, and 1992 columns called ‘Vulnerable population by year’
* Include a stubhead label heading above the countries column called 'Developer'
* Have commas to separate thousands
* Be formatted so that:
  * The spanner heading is a dark green with a HEX code of #324027 and the text is white
  * The stubhead label heading is a dark green with a HEX code of #324027 and the text is white and bold
  * The column labels are an earthy green with a HEX code of #48553F and the text is white

When you have finished making your table, compare it against the relevant code and solution in the 'exercise_solutions' subfolder. If you have followed all the guidance correctly, this is approximately how your output should look.
